# 05 - Table 表格块创建测试（最终版）

> 基于 2026-04-19 全面实测的最终结论。

## 核心结论

| 操作 | 结果 | 说明 |
|------|------|------|
| 创建 table（block_type: 31）只传 `property` | ✅ 成功 | 飞书自动生成空 cell blocks |
| POST children 到 table | ❌ 失败 | `1770028` block not support create children |
| 直接创建 table_cell（block_type: 32） | ❌ 失败 | `1770029` block not support to create |
| PATCH 更新 cell / text / heading | ❌ 失败 | `1770001` invalid param（权限不足） |
| DELETE 删除 block | ❌ 失败 | `404` page not found |
| **POST children 到 table_cell** | **✅ 成功** | **唯一可行的 cell 内容填充方式** |
| blocks/convert API | ❌ 失败 | `99991672` 缺少 docx:document.block:convert 权限 |

## 正确创建流程（两步）

### 步骤 1：创建空 table
```
POST /docx/v1/documents/{doc_id}/blocks/{doc_id}/children
{
  "children": [{
    "block_type": 31,
    "table": {
      "property": {
        "column_size": 3,
        "row_size": 2
      }
    }
  }]
}
```

返回：`table.cells` = cell block_id 数组（行优先）

### 步骤 2：POST text block 到每个 cell
```
POST /docx/v1/documents/{doc_id}/blocks/{cell_id}/children
{
  "children": [{
    "block_type": 2,
    "text": {
      "elements": [
        {"text_run": {"content": "单元格内容"}}
      ]
    }
  }]
}
```

支持行内格式：`text_element_style` 中可设 `bold` / `italic` / `strikethrough` / `inline_code` / `link` 等。

## 已知限制

1. **空 child 副作用**：每个 cell 有一个飞书自动生成的空 text block（content: ""），POST 后 cell 有两个 children，raw_content 中会多一个空行。
2. **API 请求数**：创建 N×M 表格需要 `1 + N×M` 次请求（创建 table + 每个 cell POST）。
3. **QPS 限制**：飞书限制 3 QPS，大表格需要延时（建议每 3 个请求延时 400ms）。
4. **不可更新/删除**：PATCH 和 DELETE 不可用，已创建的 table cell 内容无法修改或删除。


In [6]:
import os
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from feishu_client import FeishuClient

load_dotenv(Path('../../.env'))
client = FeishuClient()
print('客户端初始化成功')

客户端初始化成功


In [7]:
# 创建测试文档
doc = client.api('POST', '/docx/v1/documents', json_data={'title': 'Table 最终测试文档'})
doc_id = doc['document']['document_id']
print(f'文档ID: {doc_id}')
print(f'URL: https://feishu.cn/docx/{doc_id}')

文档ID: Vr31d4zZtoyQttxZ1LpcF1axnSe
URL: https://feishu.cn/docx/Vr31d4zZtoyQttxZ1LpcF1axnSe


In [8]:
# 步骤 1：创建空 table
table_block = {
    'block_type': 31,
    'table': {
        'property': {'column_size': 3, 'row_size': 2}
    }
}

r1 = client.request(
    'POST',
    f'/docx/v1/documents/{doc_id}/blocks/{doc_id}/children',
    json_data={'children': [table_block]}
)
print(json.dumps(r1, indent=2, ensure_ascii=False))

table_data = r1['data']['children'][0]
table_id = table_data['block_id']
cell_ids = table_data['table']['cells']
print(f'\ntable_id: {table_id}')
print(f'cell_ids ({len(cell_ids)} 个): {cell_ids}')

{
  "code": 0,
  "data": {
    "children": [
      {
        "block_id": "doxcnPzz9p2QiuO0aPlZDdK9rPd",
        "block_type": 31,
        "children": [
          "doxcnXKMgV6iavmmuGGKH8gTX9b",
          "doxcnW0Zq1DI09zxf0C4lbe1B8b",
          "doxcnBili5TPyZYd2umal4J281e",
          "doxcn50iEr7TKYOqWXOca4eAhXc",
          "doxcnB9HAY1uf7Yhg72lI3769cb",
          "doxcnjN8G8CBCxcpmE49lqdYyge"
        ],
        "parent_id": "Vr31d4zZtoyQttxZ1LpcF1axnSe",
        "table": {
          "cells": [
            "doxcnXKMgV6iavmmuGGKH8gTX9b",
            "doxcnW0Zq1DI09zxf0C4lbe1B8b",
            "doxcnBili5TPyZYd2umal4J281e",
            "doxcn50iEr7TKYOqWXOca4eAhXc",
            "doxcnB9HAY1uf7Yhg72lI3769cb",
            "doxcnjN8G8CBCxcpmE49lqdYyge"
          ],
          "property": {
            "column_size": 3,
            "column_width": [
              100,
              100,
              100
            ],
            "merge_info": [
              {
                "col_span": 1,


In [9]:
# 步骤 2：POST text block 到每个 cell（行优先）
contents = [
    '模型名称', '机制', '压缩率',
    'Llama 2', 'GQA', '8:1'
]

for idx, cid in enumerate(cell_ids):
    body = {
        'children': [{
            'block_type': 2,
            'text': {
                'elements': [{'text_run': {'content': contents[idx]}}]
            }
        }]
    }
    r = client.request(
        'POST',
        f'/docx/v1/documents/{doc_id}/blocks/{cid}/children',
        json_data=body
    )
    status = 'OK' if r.get('code') == 0 else f"FAIL: {r.get('msg')}"
    print(f'  Cell {idx} ({cid[:10]}...): {status}')
    # QPS 保护
    if idx > 0 and idx % 3 == 0:
        time.sleep(0.4)

print('\n全部完成！')

  Cell 0 (doxcnXKMgV...): OK
  Cell 1 (doxcnW0Zq1...): OK
  Cell 2 (doxcnBili5...): OK
  Cell 3 (doxcn50iEr...): OK
  Cell 4 (doxcnB9HAY...): OK
  Cell 5 (doxcnjN8G8...): OK

全部完成！


In [10]:
# 验证 raw_content
r_raw = client.request('GET', f'/docx/v1/documents/{doc_id}/raw_content')
print(r_raw.get('data', {}).get('content', 'N/A'))

Table 最终测试文档



模型名称


机制


压缩率


Llama 2


GQA


8:1

